# Notebook 01 — Data Profiling & Preparation (PROTOTYPE)

**Purpose:** Load, clean, and prepare 1-minute OHLC data for later cointegration
testing in Notebook 02.

**Scope:** Prototype — 10 hand-picked liquid tickers only. The full-run version
will replace the hardcoded ticker list with universe screening logic.

**Methodology source of truth:**
- `.agents/workflows/cointegration_methodology_spec.md`
- `.agents/workflows/implementation_checklist.md`

**Key decisions (already approved, not revisited here):**
- Use `close` price
- Use log prices
- Filter to 9:35–15:55 ET (exclude auction periods)
- Resample to 5-minute bars
- Outlier threshold: |z| > 10σ on minute returns

## 1. Setup and Configuration

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import glob
import warnings
import sys, io

# Fix Windows console encoding for Unicode characters in print()
if sys.stdout.encoding != 'utf-8':
    sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8', errors='replace')

warnings.filterwarnings('ignore', category=FutureWarning)

# ── Paths ──
PROJECT_ROOT = Path(r"d:\Quant Finance\Quant Program\Week 1")
DATA_DIR = PROJECT_ROOT / "data"
INTERMEDIATE_DIR = DATA_DIR / "intermediate"
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)

MONTH_FOLDERS = sorted([f"{i:02d}" for i in range(1, 13)])

# ── Prototype universe ──
# PROTOTYPE-ONLY: In the full run, this list comes from universe screening.
# These 10 tickers are hand-picked for known liquidity and sector diversity.
PROTOTYPE_TICKERS = [
    'AAPL',  # Technology
    'MSFT',  # Technology
    'AMZN',  # Consumer Discretionary
    'XOM',   # Energy
    'CVX',   # Energy
    'JPM',   # Financials
    'BAC',   # Financials
    'V',     # Financials (Payments)
    'MA',    # Financials (Payments)
    'GOOGL', # Technology (Communication Services) — replaced META (only 161 trading days)
]

# ── Approved methodology parameters ──
SESSION_START = pd.Timestamp('09:35:00').time()  # Inclusive — excludes opening auction
SESSION_END = pd.Timestamp('15:55:00').time()    # Inclusive — excludes closing auction
RESAMPLE_FREQ = '5min'
OUTLIER_ZSCORE_THRESHOLD = 10  # |z| > 10σ flags a data error
OUTLIER_PCT_REMOVAL_THRESHOLD = 1.0  # If > 1% of returns are outliers, remove ticker
TOTAL_OUTLIER_BUDGET_PCT = 0.5  # Total modified points must be < 0.5% of all data

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir: {DATA_DIR}")
print(f"Prototype tickers: {PROTOTYPE_TICKERS}")
print(f"Session window: {SESSION_START} – {SESSION_END} ET")
print(f"Resample frequency: {RESAMPLE_FREQ}")

## 2. Data Loading

Load minute-bar CSVs for prototype tickers across all 12 months.
Each file is `TICKER_YYYY-MM-DD.csv` with schema:
`ticker,volume,open,close,high,low,window_start,transactions`

`window_start` is nanoseconds since Unix epoch (UTC).

In [ ]:
def load_ticker_data(ticker: str, data_dir: Path, month_folders: list[str]) -> pd.DataFrame:
    """Load all daily CSV files for a single ticker across all months.

    Returns a DataFrame with columns [close, volume] indexed by ET datetime.
    Returns empty DataFrame if no files found.
    """
    all_frames = []
    files_found = 0

    for month in month_folders:
        month_dir = data_dir / month
        pattern = str(month_dir / f"{ticker}_*.csv")
        files = glob.glob(pattern)
        for fpath in files:
            try:
                df = pd.read_csv(fpath, usecols=['close', 'volume', 'window_start'])
                all_frames.append(df)
                files_found += len(df)
            except Exception as e:
                print(f"  WARNING: Failed to read {fpath}: {e}")

    if not all_frames:
        return pd.DataFrame(columns=['close', 'volume'])

    combined = pd.concat(all_frames, ignore_index=True)

    # Convert nanosecond UTC timestamps to ET DatetimeIndex
    combined['datetime_et'] = (
        pd.to_datetime(combined['window_start'], unit='ns', utc=True)
        .dt.tz_convert('US/Eastern')
    )
    combined = combined.set_index('datetime_et').sort_index()

    # Drop duplicate timestamps (keep last per checklist)
    n_dupes = combined.index.duplicated(keep='last').sum()
    if n_dupes > 0:
        combined = combined[~combined.index.duplicated(keep='last')]

    combined = combined[['close', 'volume']]
    return combined


# ── Load all prototype tickers ──
raw_data = {}
load_summary = []

for ticker in PROTOTYPE_TICKERS:
    print(f"Loading {ticker}...", end=" ")
    df = load_ticker_data(ticker, DATA_DIR, MONTH_FOLDERS)
    n_rows = len(df)
    raw_data[ticker] = df

    if n_rows == 0:
        print("NO DATA FOUND")
        load_summary.append({'ticker': ticker, 'n_raw_minutes': 0, 'status': 'NO DATA'})
    else:
        first_ts = df.index[0]
        last_ts = df.index[-1]
        print(f"{n_rows:,} rows | {first_ts.date()} to {last_ts.date()}")
        load_summary.append({
            'ticker': ticker,
            'n_raw_minutes': n_rows,
            'first_timestamp': str(first_ts),
            'last_timestamp': str(last_ts),
            'status': 'OK'
        })

load_summary_df = pd.DataFrame(load_summary)
print(f"\n{'='*60}")
print(f"Loaded {len([t for t in raw_data if len(raw_data[t]) > 0])}/{len(PROTOTYPE_TICKERS)} tickers successfully")

# ── Handle any missing tickers ──
missing_tickers = [t for t in PROTOTYPE_TICKERS if len(raw_data[t]) == 0]
if missing_tickers:
    print(f"\nWARNING: No data for: {missing_tickers}")
    print("These tickers will be excluded from the prototype.")
    PROTOTYPE_TICKERS = [t for t in PROTOTYPE_TICKERS if t not in missing_tickers]

### 2a. Raw data inspection

In [ ]:
# Show raw schema and sample rows for AAPL
sample_ticker = 'AAPL'
print(f"Raw data sample for {sample_ticker}:")
print(f"  Shape: {raw_data[sample_ticker].shape}")
print(f"  Index dtype: {raw_data[sample_ticker].index.dtype}")
print(f"  Index tz: {raw_data[sample_ticker].index.tz}")
print(f"  Columns: {list(raw_data[sample_ticker].columns)}")
print()
raw_data[sample_ticker].head(10)

## 3. Timestamp Conversion and Session Filtering

**Approved rule:** Keep only 9:35–15:55 ET (inclusive on both ends).
This excludes pre-market, after-hours, the first 5 minutes of the open
(auction effects), and the last 5 minutes before close (closing auction).

In [ ]:
def filter_session(df: pd.DataFrame) -> pd.DataFrame:
    """Filter DataFrame to approved trading session 9:35-15:55 ET.

    Assumes index is timezone-aware ET DatetimeIndex.
    """
    times = df.index.time
    mask = (times >= SESSION_START) & (times <= SESSION_END)
    return df.loc[mask]


# ── Apply session filter to all tickers ──
filtered_data = {}
filter_report = []

for ticker in PROTOTYPE_TICKERS:
    raw_df = raw_data[ticker]
    n_before = len(raw_df)

    filt_df = filter_session(raw_df)
    n_after = len(filt_df)
    n_dropped = n_before - n_after

    filtered_data[ticker] = filt_df
    pct_kept = (n_after / n_before * 100) if n_before > 0 else 0

    filter_report.append({
        'ticker': ticker,
        'n_raw_minutes': n_before,
        'n_filtered_minutes': n_after,
        'n_dropped': n_dropped,
        'pct_kept': round(pct_kept, 1),
    })

filter_report_df = pd.DataFrame(filter_report)
print("Session filter results (9:35–15:55 ET):")
print(filter_report_df.to_string(index=False))

### 3a. Verify session filter correctness

In [ ]:
# Spot-check: verify time boundaries on a January day (EST) and July day (EDT)
# Use tz-aware timestamps for slicing since the index is tz-aware
check_dates = {
    '2022-01-04': 'EST',  # A Tuesday in January (markets open)
    '2022-07-05': 'EDT',  # A Tuesday in July
}

for check_ticker in ['AAPL', 'MSFT']:
    filt = filtered_data[check_ticker]
    for date_str, tz_label in check_dates.items():
        day_mask = filt.index.date == pd.Timestamp(date_str).date()
        day_data = filt[day_mask]
        if len(day_data) > 0:
            day_min = day_data.index.min().time()
            day_max = day_data.index.max().time()
            print(f"{check_ticker} {date_str} ({tz_label}): {len(day_data)} bars, "
                  f"first={day_min}, last={day_max}")
            assert day_min >= SESSION_START, f"Time filter failed: {day_min} < {SESSION_START}"
            assert day_max <= SESSION_END, f"Time filter failed: {day_max} > {SESSION_END}"
        else:
            print(f"{check_ticker} {date_str}: no data (may not be a trading day)")

print("\nSession filter verification PASSED for both EST and EDT days.")

# Check sorted index
for ticker in PROTOTYPE_TICKERS:
    assert filtered_data[ticker].index.is_monotonic_increasing, \
        f"{ticker} index is not sorted!"
print("All ticker indices are sorted and monotonically increasing.")

## 4. Per-Ticker Data Audit

Compute profiling statistics required by the implementation checklist:
- Number of observations, date coverage
- Missingness / completeness
- Duplicate timestamps (already handled during load)
- Price sanity checks (median price, zero-return fraction)
- Dollar volume

In [ ]:
def compute_ticker_profile(ticker: str, df: pd.DataFrame) -> dict:
    """Compute data quality profile for a single ticker.

    df: session-filtered DataFrame with columns [close, volume].
    """
    if len(df) == 0:
        return {'ticker': ticker, 'n_filtered_minutes': 0, 'status': 'EMPTY'}

    # Date coverage
    trading_dates = df.index.normalize().unique()
    n_trading_days = len(trading_dates)
    first_date = trading_dates.min().date()
    last_date = trading_dates.max().date()

    # Completeness: expected 381 minutes per full trading day
    # (9:35 to 15:55 inclusive = 381 one-minute bars on a full session day)
    expected_minutes = n_trading_days * 381
    actual_minutes = len(df)
    completeness_pct = (actual_minutes / expected_minutes * 100) if expected_minutes > 0 else 0

    # Price stats
    median_close = df['close'].median()
    min_close = df['close'].min()
    max_close = df['close'].max()

    # Dollar volume: per-day sum of (volume * close), then mean across days
    df_with_dolv = df.copy()
    df_with_dolv['dollar_volume'] = df_with_dolv['volume'] * df_with_dolv['close']
    daily_dollar_vol = df_with_dolv.groupby(df_with_dolv.index.date)['dollar_volume'].sum()
    avg_daily_dollar_volume = daily_dollar_vol.mean()

    # Zero-return fraction (minute-level)
    returns = df['close'].pct_change()
    # Exclude first return of each day (NaN from overnight gap)
    day_starts = df.index.to_series().diff() > pd.Timedelta(minutes=5)
    returns[day_starts] = np.nan
    valid_returns = returns.dropna()
    zero_return_pct = ((valid_returns == 0).sum() / len(valid_returns) * 100) if len(valid_returns) > 0 else 0

    return {
        'ticker': ticker,
        'n_filtered_minutes': actual_minutes,
        'n_trading_days': n_trading_days,
        'first_date': str(first_date),
        'last_date': str(last_date),
        'completeness_pct': round(completeness_pct, 1),
        'median_close': round(median_close, 2),
        'min_close': round(min_close, 2),
        'max_close': round(max_close, 2),
        'avg_daily_dollar_volume': round(avg_daily_dollar_volume, 0),
        'zero_return_pct': round(zero_return_pct, 1),
    }


# ── Profile all prototype tickers ──
profiles = []
for ticker in PROTOTYPE_TICKERS:
    profile = compute_ticker_profile(ticker, filtered_data[ticker])
    profiles.append(profile)

profile_df = pd.DataFrame(profiles)
print("Per-Ticker Data Audit:")
print(profile_df.to_string(index=False))

### 4a. Screening check (prototype — informational only)

In the full run, tickers failing these thresholds would be removed.
For the prototype, all 10 tickers are known-liquid, so we expect all to pass.
We still run the checks to validate the screening logic.

In [ ]:
# Screening thresholds (from approved methodology)
SCREEN_MIN_MEDIAN_PRICE = 5.0
SCREEN_MIN_AVG_DAILY_DOLV = 1_000_000
SCREEN_MIN_COMPLETENESS = 90.0
SCREEN_MAX_ZERO_RETURN_PCT = 50.0

screening_results = []
for _, row in profile_df.iterrows():
    reason = None
    if row['median_close'] < SCREEN_MIN_MEDIAN_PRICE:
        reason = f"median_close={row['median_close']} < ${SCREEN_MIN_MEDIAN_PRICE}"
    elif row['avg_daily_dollar_volume'] < SCREEN_MIN_AVG_DAILY_DOLV:
        reason = f"avg_daily_dolv={row['avg_daily_dollar_volume']:,.0f} < ${SCREEN_MIN_AVG_DAILY_DOLV:,.0f}"
    elif row['completeness_pct'] < SCREEN_MIN_COMPLETENESS:
        reason = f"completeness={row['completeness_pct']}% < {SCREEN_MIN_COMPLETENESS}%"
    elif row['zero_return_pct'] >= SCREEN_MAX_ZERO_RETURN_PCT:
        reason = f"zero_return={row['zero_return_pct']}% >= {SCREEN_MAX_ZERO_RETURN_PCT}%"

    screening_results.append({
        'ticker': row['ticker'],
        'passed_screening': reason is None,
        'rejection_reason': reason if reason else '',
    })

screening_df = pd.DataFrame(screening_results)
n_passed = screening_df['passed_screening'].sum()
n_failed = len(screening_df) - n_passed

print(f"\nScreening results: {n_passed} passed, {n_failed} failed")
if n_failed > 0:
    print("Failed tickers:")
    print(screening_df[~screening_df['passed_screening']].to_string(index=False))

# For prototype: keep all tickers that pass screening
surviving_tickers = screening_df[screening_df['passed_screening']]['ticker'].tolist()
print(f"\nSurviving tickers for prototype: {surviving_tickers}")

## 5. Cleaning: Outlier Treatment

**Approved rule:** Flag minute returns with |z| > 10σ. Replace the corresponding
close price with NaN, then forward-fill (max 1 bar).

**Transparency:** Log every modification per ticker. If any ticker exceeds 1%
outlier rate, remove it entirely.

In [ ]:
outlier_log = []
cleaned_data = {}
total_points = 0
total_outliers = 0

for ticker in surviving_tickers:
    df = filtered_data[ticker][['close', 'volume']].copy()
    n_total = len(df)
    total_points += n_total

    # Compute minute returns
    returns = df['close'].pct_change()

    # Exclude day boundaries from z-score calculation
    day_starts = df.index.to_series().diff() > pd.Timedelta(minutes=5)
    returns[day_starts] = np.nan

    valid_returns = returns.dropna()
    if len(valid_returns) < 100:
        # Not enough data to compute meaningful z-scores
        outlier_log.append({
            'ticker': ticker, 'n_total': n_total,
            'n_outliers_flagged': 0, 'pct_outliers': 0.0,
            'action': 'kept (too few returns for z-score)',
        })
        cleaned_data[ticker] = df
        continue

    ret_mean = valid_returns.mean()
    ret_std = valid_returns.std()

    if ret_std == 0:
        outlier_log.append({
            'ticker': ticker, 'n_total': n_total,
            'n_outliers_flagged': 0, 'pct_outliers': 0.0,
            'action': 'kept (zero std)',
        })
        cleaned_data[ticker] = df
        continue

    z_scores = (returns - ret_mean) / ret_std
    outlier_mask = z_scores.abs() > OUTLIER_ZSCORE_THRESHOLD
    n_outliers = outlier_mask.sum()
    pct_outliers = n_outliers / n_total * 100
    total_outliers += n_outliers

    if pct_outliers > OUTLIER_PCT_REMOVAL_THRESHOLD:
        # Hard rule: remove ticker entirely
        outlier_log.append({
            'ticker': ticker, 'n_total': n_total,
            'n_outliers_flagged': int(n_outliers), 'pct_outliers': round(pct_outliers, 3),
            'action': f'REMOVED (>{OUTLIER_PCT_REMOVAL_THRESHOLD}% outliers)',
        })
        continue  # Do not add to cleaned_data

    # Replace outlier close prices with NaN, then ffill (max 1 bar)
    if n_outliers > 0:
        df.loc[outlier_mask, 'close'] = np.nan
        df['close'] = df['close'].ffill(limit=1)

    outlier_log.append({
        'ticker': ticker, 'n_total': n_total,
        'n_outliers_flagged': int(n_outliers), 'pct_outliers': round(pct_outliers, 4),
        'action': f'kept ({n_outliers} points patched)' if n_outliers > 0 else 'kept (clean)',
    })
    cleaned_data[ticker] = df

outlier_df = pd.DataFrame(outlier_log)
print("Outlier Treatment Transparency Report:")
print(outlier_df.to_string(index=False))

# Check total outlier budget
total_outlier_pct = (total_outliers / total_points * 100) if total_points > 0 else 0
print(f"\nTotal: {total_outliers:,} outliers flagged out of {total_points:,} total points "
      f"({total_outlier_pct:.4f}%)")
if total_outlier_pct > TOTAL_OUTLIER_BUDGET_PCT:
    print(f"WARNING: Total outlier rate {total_outlier_pct:.2f}% exceeds budget "
          f"of {TOTAL_OUTLIER_BUDGET_PCT}%. Investigation needed.")
else:
    print(f"Total outlier rate within budget (<{TOTAL_OUTLIER_BUDGET_PCT}%).")

# Update surviving tickers list (in case any were removed)
surviving_tickers = list(cleaned_data.keys())
removed_by_outliers = set(screening_df[screening_df['passed_screening']]['ticker']) - set(surviving_tickers)
if removed_by_outliers:
    print(f"\nTickers removed due to high outlier rate: {removed_by_outliers}")
print(f"\nFinal surviving tickers: {surviving_tickers} ({len(surviving_tickers)} total)")

## 6. Log Transform and Resampling

**Approved rule:**
1. Assert all prices > 0 (guaranteed by $5 median filter, but verify)
2. Apply `np.log(close)`
3. Resample to 5-minute bars using last value in each 5-min window

In [ ]:
# ── Step 1: Assert positive prices ──
for ticker in surviving_tickers:
    close_series = cleaned_data[ticker]['close'].dropna()
    assert (close_series > 0).all(), \
        f"{ticker} has non-positive prices! Min={close_series.min()}"
print("Price positivity check PASSED for all tickers.")

# ── Step 2: Log transform ──
log_data = {}
for ticker in surviving_tickers:
    df = cleaned_data[ticker].copy()
    df['log_close'] = np.log(df['close'])
    log_data[ticker] = df

# ── Step 3: Resample to 5-minute bars ──
resampled = {}
resample_report = []

for ticker in surviving_tickers:
    df = log_data[ticker]
    n_before = len(df)

    # Resample: take last log_close in each 5-min window
    log_5min = df['log_close'].resample(RESAMPLE_FREQ).last()

    # Drop NaN rows (from empty 5-min windows, e.g., overnight gaps)
    log_5min = log_5min.dropna()

    n_after = len(log_5min)
    resampled[ticker] = log_5min

    resample_report.append({
        'ticker': ticker,
        'n_1min_bars': n_before,
        'n_5min_bars': n_after,
        'ratio': round(n_before / n_after, 1) if n_after > 0 else 0,
    })

resample_df = pd.DataFrame(resample_report)
print("Resample report (1-min -> 5-min):")
print(resample_df.to_string(index=False))
print(f"\nExpected ratio ≈ 5.0 (5 one-min bars per 5-min window)")

### 6a. Align tickers on a common timestamp grid

In [ ]:
# Build the aligned panel: columns = tickers, index = 5-min timestamps
# Use inner join so every row has data for ALL tickers
log_price_panel = pd.DataFrame(resampled)

# Check for NaN before dropping
n_rows_before = len(log_price_panel)
nan_counts = log_price_panel.isna().sum()
print("NaN counts per ticker before alignment:")
print(nan_counts)

# Inner join: keep only rows where ALL tickers have data
log_price_panel = log_price_panel.dropna()
n_rows_after = len(log_price_panel)
n_dropped = n_rows_before - n_rows_after

print(f"\nAlignment: {n_rows_before} → {n_rows_after} rows "
      f"({n_dropped} dropped, {n_dropped/n_rows_before*100:.1f}% loss)")

# Verify shape
print(f"\nFinal panel shape: {log_price_panel.shape}")
print(f"  Rows (5-min bars): {log_price_panel.shape[0]}")
print(f"  Columns (tickers): {log_price_panel.shape[1]}")
print(f"  Date range: {log_price_panel.index[0]} to {log_price_panel.index[-1]}")

# Sanity: verify log prices by inverting a sample
sample_ticker = surviving_tickers[0]
sample_log = log_price_panel[sample_ticker].iloc[0]
sample_price = np.exp(sample_log)
print(f"\nSanity check: {sample_ticker} first log_close={sample_log:.6f} → "
      f"exp()=${sample_price:.2f}")

## 7. Output Artifacts

Save the cleaned aligned panel and metadata for Notebook 02.

In [ ]:
# ── Save main data artifact ──
output_path = INTERMEDIATE_DIR / "log_prices_5min.parquet"
log_price_panel.to_parquet(output_path, engine='pyarrow')
print(f"Saved: {output_path}")
print(f"  Shape: {log_price_panel.shape}")
print(f"  Size: {output_path.stat().st_size / 1024:.1f} KB")

# ── Build and save the Data Audit Summary table (Table 1 Part B from checklist) ──
audit_records = []
for ticker in PROTOTYPE_TICKERS:
    # Get profile data
    prof_row = profile_df[profile_df['ticker'] == ticker].iloc[0] if ticker in profile_df['ticker'].values else None
    screen_row = screening_df[screening_df['ticker'] == ticker].iloc[0] if ticker in screening_df['ticker'].values else None
    outlier_row = outlier_df[outlier_df['ticker'] == ticker].iloc[0] if ticker in outlier_df['ticker'].values else None
    resamp_row = resample_df[resample_df['ticker'] == ticker].iloc[0] if ticker in resample_df['ticker'].values else None

    audit_records.append({
        'ticker': ticker,
        'n_raw_minutes': int(prof_row['n_filtered_minutes']) if prof_row is not None else 0,
        'n_filtered_minutes': int(prof_row['n_filtered_minutes']) if prof_row is not None else 0,
        'n_5min_bars': int(resamp_row['n_5min_bars']) if resamp_row is not None else 0,
        'median_close': float(prof_row['median_close']) if prof_row is not None else np.nan,
        'avg_daily_dollar_volume': float(prof_row['avg_daily_dollar_volume']) if prof_row is not None else np.nan,
        'completeness_pct': float(prof_row['completeness_pct']) if prof_row is not None else np.nan,
        'zero_return_pct': float(prof_row['zero_return_pct']) if prof_row is not None else np.nan,
        'n_outliers_flagged': int(outlier_row['n_outliers_flagged']) if outlier_row is not None else 0,
        'pct_outliers': float(outlier_row['pct_outliers']) if outlier_row is not None else 0.0,
        'passed_screening': bool(screen_row['passed_screening']) if screen_row is not None else False,
        'rejection_reason': str(screen_row['rejection_reason']) if screen_row is not None else 'no data',
    })

audit_summary = pd.DataFrame(audit_records)

# Save as parquet for Notebook 02
audit_path = INTERMEDIATE_DIR / "universe_metadata.parquet"
audit_summary.to_parquet(audit_path, engine='pyarrow')
print(f"\nSaved: {audit_path}")

print("\n── Data Audit Summary Table (Table 1 Part B) ──")
print(audit_summary.to_string(index=False))

### Ticker Screening Summary

In [ ]:
print("── Ticker Screening Summary ──")
print(f"Total prototype tickers attempted: {len(PROTOTYPE_TICKERS)}")
print(f"Passed screening: {len(surviving_tickers)}")
print(f"In final aligned panel: {log_price_panel.shape[1]}")
print()

# Merge screening + outlier info for a compact view
screen_summary = screening_df.merge(
    outlier_df[['ticker', 'n_outliers_flagged', 'pct_outliers', 'action']],
    on='ticker', how='left'
)
print(screen_summary.to_string(index=False))

## 8. Validation Cells

In [ ]:
# ── Validation 1: Parquet reloads correctly ──
print("Validation 1: Parquet reload test")
reloaded = pd.read_parquet(output_path, engine='pyarrow')
assert reloaded.shape == log_price_panel.shape, \
    f"Shape mismatch: saved {log_price_panel.shape}, loaded {reloaded.shape}"
assert list(reloaded.columns) == list(log_price_panel.columns), \
    "Column mismatch after reload"

# Check timezone preservation
if reloaded.index.tz is None:
    print("  NOTE: Parquet stripped timezone. Re-localizing to US/Eastern after reload.")
    print("  (Notebook 02 must handle this on load.)")
else:
    print(f"  Index timezone preserved: {reloaded.index.tz}")

print(f"  Shape: {reloaded.shape} ✓")
print(f"  First timestamp: {reloaded.index[0]} ✓")
print(f"  Last timestamp: {reloaded.index[-1]} ✓")
print()

# ── Validation 2: No duplicate timestamps ──
print("Validation 2: Duplicate timestamp check")
n_dupes = reloaded.index.duplicated().sum()
assert n_dupes == 0, f"Found {n_dupes} duplicate timestamps!"
print(f"  Duplicates: {n_dupes} ✓")
print()

# ── Validation 3: No NaN in panel ──
print("Validation 3: NaN check")
total_nan = reloaded.isna().sum().sum()
assert total_nan == 0, f"Found {total_nan} NaN values in aligned panel!"
print(f"  Total NaN: {total_nan} ✓")
print()

# ── Validation 4: Data types ──
print("Validation 4: Data type check")
for col in reloaded.columns:
    assert reloaded[col].dtype == np.float64, \
        f"Column {col} has dtype {reloaded[col].dtype}, expected float64"
print(f"  All {len(reloaded.columns)} columns are float64 ✓")
print()

# ── Validation 5: Log price sanity (exp should give reasonable stock prices) ──
print("Validation 5: Log price sanity")
for col in reloaded.columns:
    prices = np.exp(reloaded[col])
    p_min, p_max = prices.min(), prices.max()
    assert p_min > 1, f"{col}: exp(log_close) min={p_min:.2f} — suspiciously low"
    assert p_max < 10000, f"{col}: exp(log_close) max={p_max:.2f} — suspiciously high"
    print(f"  {col}: ${p_min:.2f} – ${p_max:.2f} ✓")

## Prototype Status Summary

In [ ]:
print("=" * 60)
print("NOTEBOOK 01 PROTOTYPE — STATUS SUMMARY")
print("=" * 60)
print()
print(f"Tickers loaded:          {len(PROTOTYPE_TICKERS)}")
print(f"Tickers after screening: {len(surviving_tickers)}")
print(f"Tickers in final panel:  {log_price_panel.shape[1]}")
print()
print(f"Session filter:          9:35–15:55 ET")
print(f"Resample frequency:      5-minute bars")
print(f"Price transform:         log(close)")
print()
print(f"Final panel shape:       {log_price_panel.shape[0]} rows × {log_price_panel.shape[1]} columns")
print(f"Date range:              {log_price_panel.index[0].date()} to {log_price_panel.index[-1].date()}")
print(f"NaN in panel:            0")
print(f"Duplicate timestamps:    0")
print()
print(f"Outliers flagged total:  {total_outliers}")
print(f"Outlier rate:            {total_outlier_pct:.4f}%")
print()
print("Output files:")
print(f"  {output_path}")
print(f"  {audit_path}")
print()
print("PROTOTYPE READY FOR NOTEBOOK 02.")

### Assumptions and Notes

1. **Prototype only:** This notebook loads 10 hand-picked tickers. The full-run
   version will replace this with the complete universe screening pipeline
   (317 tickers → quality filters → top-50 cap).

2. **Price adjustment:** We assume the flat-file prices are already split-adjusted
   and dividend-adjusted. The data documentation does not explicitly confirm this.

3. **Timezone:** `window_start` is UTC nanoseconds, converted to US/Eastern.
   Pandas handles EST/EDT transitions automatically.

4. **Parquet timezone:** Some parquet engines strip timezone info on save.
   Notebook 02 should re-localize the index to US/Eastern if timezone is None
   after reload.

5. **No pair generation or cointegration testing in this notebook.**
   That is Notebook 02's responsibility.